# MuseTalk 환경 구축 (Colab)

**왜 만들었나:** AI Review Board 프로젝트에서 위원회 아바타(A/B 2인)가 발언 텍스트를 립싱크 영상으로 말하게 하는 기능을 위해, MuseTalk을 Colab A100 GPU 위에서 돌아가게 세팅하는 노트북입니다. (작업분해표 1~2, 3, 5단계에 대응)

**중요 — Python 버전 문제:** Colab 기본 Python은 3.12인데, `mmcv==2.0.1`(MuseTalk이 요구하는 버전)은 Python 3.12용 사전 컴파일 wheel이 아예 제공되지 않습니다(3.8~3.11까지만 지원). 그래서 Colab 기본 환경에 바로 설치를 시도하면 `mim`/`pip` 어떤 방법을써도 실패합니다. 이 문제를 피하려고 **Miniconda로 Python 3.10 가상환경을 따로 만들어 그 안에서 전부 설치·실행**합니다. (2차 프로젝트 로컬 환경도 공식 문서대로 `python==3.10`이었기 때문에, 이 버전으로 맞추면 검증된 조합을 그대로 쓸 수 있습니다.)

**이 노트북이 하는 일 (순서대로):**
1. Miniconda 설치 + Python 3.10 가상환경(`musetalk`) 생성
2. MuseTalk 공식 레포 클론
3. GPU(A100) 인식 확인
4. 의존성(pip 패키지 + ffmpeg + mmengine/mmcv/mmdet/mmpose) 설치 — 전부 `musetalk` 가상환경 안에서
5. 모델 가중치를 Google Drive에서 마운트해서 가져오기 (Hugging Face에서 로컬 PC로 미리 받아 Drive에 올려둔 것 사용)
6. 샘플 추론 테스트 — 공식 예제 오디오+영상으로 1회 정상 동작 확인
7. 페르소나 A 영상 업로드 + TTS(OpenAI `gpt-4o-mini-tts`, `ai/media/tts/voice_map.json` 반영)로 실제 발언 문장 생성
8. TTS 구간만 립싱크 추론 (`--extra_margin 15`)
9. (필요시) 입 주변 색보정 + 프레임 시간축 스무딩으로 우글거림 완화
10. 무음 tail(원본 루프 영상)을 이어붙여 10초 단위로 영상 길이 고정 — 품질 실험용, API에는 아직 미적용
11~14. **FastAPI로 감싸서 cloudflared로 외부 노출, 실제 HTTP 요청으로 E2E 테스트** (지금은 TTS+립싱크만, 품질 후처리는 나중에 다시 붙임)

**주의:** Colab 세션이 종료/재시작되면 conda 가상환경 자체도 날아갑니다 (Colab VM이 초기화되므로). 세션이 새로 시작되면 1번부터 다시 실행해야 합니다 (모델 파일은 Drive에 있으므로 재다운로드는 필요 없음). cloudflared Quick Tunnel URL도 재실행마다 바뀝니다. OpenAI API 키는 Colab 보안 비밀(Secrets)에 `OPENAI_API_KEY`로 등록되어 있어야 합니다.

## 1. Miniconda 설치 + Python 3.10 가상환경 생성
이후 모든 명령어는 `source .../conda.sh && conda activate musetalk && <명령어>` 형태로 이 가상환경 안에서 실행합니다.

In [ ]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /usr/local/miniconda

# Anaconda 기본 채널 이용약관 동의 (최근 정책 변경으로 필수)
!/usr/local/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/usr/local/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!/usr/local/miniconda/bin/conda create -n musetalk python=3.10 -y

## 2. MuseTalk 레포 클론
공식 GitHub 레포에서 추론 코드(모델 아키텍처, 스크립트 등)만 받아옵니다. 모델 가중치는 여기 포함되어 있지 않습니다.

In [ ]:
!git clone https://github.com/TMElyralab/MuseTalk.git
%cd MuseTalk

## 3. GPU 동작 확인
`nvidia-smi`는 특정 Python 환경과 무관하게 Colab VM 자체의 GPU 할당 여부를 보여줍니다. A100 정보가 출력되어야 정상입니다. 안 뜨면 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(A100)** 로 변경 후 다시 실행하세요.

In [ ]:
!nvidia-smi

## 4. 의존성 설치 (musetalk 가상환경 안에서)
Python 3.10 환경이라 `mmcv` 사전 컴파일 wheel 문제, `mim`의 Python 3.12 비호환(`pkgutil.ImpImporter`) 문제가 애초에 발생하지 않습니다. 버전은 2차 프로젝트(RTX 4060)에서 실제로 성공했던 조합을 그대로 사용합니다: `torch==2.0.1+cu118`, `mmengine==0.10.7`, `mmcv==2.0.1`, `mmdet==3.1.0`, `mmpose==1.1.0`.

In [ ]:
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
pip install -r requirements.txt
!apt-get install -y ffmpeg

In [ ]:
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
pip install -U openmim && \
mim install mmengine==0.10.7 && \
mim install "mmcv==2.0.1" && \
mim install "mmdet==3.1.0"

# mmpose의 의존 패키지 chumpy는 오래된 setup.py라 pip 기본 격리 빌드 환경에 numpy가 없어 실패함.
# 이미 설치된 numpy를 그대로 쓰도록 --no-build-isolation으로 chumpy를 먼저 깔아줌
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
pip install --no-build-isolation chumpy && \
mim install "mmpose==1.1.0"

In [ ]:
# 설치 확인
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
python -c "import torch, mmcv, mmdet, mmpose; print('torch', torch.__version__, torch.cuda.is_available()); print('mmcv', mmcv.__version__); print('mmpose OK')"

## 5. 모델 가중치 가져오기 (Google Drive 경유)

로컬 PC에서 Hugging Face로부터 미리 받아둔 모델 파일들을 Google Drive의 `MuseTalk_models/` 폴더에 아래 구조로 업로드해 두었다는 전제입니다. Drive 마운트는 Python 환경과 무관한 파일시스템 레벨 작업이라 가상환경 활성화가 필요 없습니다.

```
MuseTalk_models/
├── musetalkV15/         (musetalk.json, unet.pth)
├── sd-vae/              (config.json, diffusion_pytorch_model.bin)
├── whisper/             (config.json, preprocessor_config.json, pytorch_model.bin)
├── dwpose/              (dw-ll_ucoco_384.pth)
├── syncnet/             (latentsync_syncnet.pt)
└── face-parse-bisent/   (79999_iter.pth, resnet18-5c106cde.pth)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p ./models
!cp -r "/content/drive/MyDrive/MuseTalk_models"/* ./models/

In [ ]:
!ls -R ./models

## 6. 샘플 추론 테스트
공식 예제(입력 비디오+오디오, `configs/inference/test.yaml`)로 1회 추론을 돌려 정상 동작을 확인합니다. `musetalk` 가상환경 안에서 실행해야 합니다. 여기서 에러가 나면 이후 단계(Base 영상 준비, 립싱크 파이프라인 등)가 전부 막히므로 반드시 성공 확인 후 다음으로 넘어가야 합니다.

In [ ]:
# MPLBACKEND가 Colab 기본 커널의 인라인 백엔드로 상속되어 matplotlib이 에러를 내므로 Agg(headless)로 강제 지정
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
MPLBACKEND=Agg sh inference.sh v1.5 normal

## 6-1. MuseTalk 소스 패치 — 스트리밍 첫 청크 지연 원인 제거 + 페르소나 준비 캐싱

`/generate-stream`에서 `[TIMING]` 로그로 정확히 측정한 결과, 첫 청크가 나가기까지 걸린 약 74초 중 **69.27초가 "PNG 프레임 파일이 디스크에 하나도 없는" 시간**이었습니다.

**1차 시도(배치 인터리브)**: "전체 추론 먼저 → 그다음 전체 저장" 2단계 구조를 배치 단위로 합침. 결과: 69.27s → 66.72s로 거의 개선 안 됨 — 진짜 병목은 배치 루프 진입 **이전**(47.42초)에 있었습니다.

**2차 시도(페르소나 준비 캐싱, 실측 완료)**: `frame_list`/`coord_list`/`input_latent_list`(같은 원본 영상이면 TTS 문장과 무관하게 항상 동일)를 페르소나별로 한 번만 계산해서 파일로 캐싱. **실측 결과**:

| | 캐시 미스(1번째, 캐시 생성) | 캐시 히트(2번째) |
|---|---|---|
| 배치 루프 진입 전 시간 | 53.74s | **17.96s** |
| 첫 프레임 파일 발견(경과) | 74.58s | **37.1s** |
| 첫 청크 전송 | 78.01s | **40.5s** |

첫 청크까지 걸리는 시간이 **거의 절반으로 줄었습니다** (78s → 40.5s). 단, 캐시가 없는 최초 1회 요청은 여전히 느립니다(캐시를 만드는 요청이라 정상).

**3차 시도(이번, 지연 import)**: 캐시 히트 상황에서도 `main() 시작`과 실제 배치 루프 사이에 예상보다 큰 시간차가 있었는데, 원인은 `scripts/inference.py`가 파일 맨 위에서 `musetalk.utils.preprocessing`을 무조건 import한다는 것이었습니다. 이 모듈은 **import되는 순간 모듈 최상단 코드가 실행되면서 DWPose 포즈 모델 + 얼굴 감지 모델을 GPU에 자동으로 로드**합니다:

```python
model = init_model(config_file, checkpoint_file, device=device)  # DWPose
fa = FaceAlignment(LandmarksType._2D, flip_input=False, device=device)  # 얼굴 감지
```

캐시 히트 시에는 랜드마크를 재계산할 일이 없어서 이 두 모델을 아예 쓰지 않는데도, import 시점에 무조건 GPU에 올라갔던 것입니다. 이 import를 **캐시 미스일 때만 실행되는 지연(lazy) import**로 옮겼습니다 — 캐시 히트 요청은 이제 이 두 모델을 아예 로드하지 않습니다.

**주의**: 이 패치는 `/content/MuseTalk/scripts/inference.py` 파일 자체를 덮어쓰는 것이라, Colab 런타임이 재시작되면(레포를 다시 클론하면) 함께 사라집니다. 매 세션마다 2번(레포 클론) 이후에는 이 셀을 한 번 실행해야 적용됩니다. 캐시 파일(`results/*_full_prep_*.pt`)도 세션이 끝나면 같이 사라지므로, 세션마다 첫 요청은 항상 느리다는 점을 감안하세요.

**검증 방법**: 이 셀 실행 후 서버를 재시작(`lsof -i:8000` → `kill -9 <PID>` → 12번 → 13번)하고, **같은 speaker_id로 스트리밍 테스트를 연속 2번** 돌려보세요. 2번째 요청(캐시 히트)에서 `[INF-TIMING]`에 `DWPose/얼굴 감지 모델 로드 완료` 줄이 **나타나지 않아야** 정상이고(캐시 히트라 스킵), `전체 준비 캐시 로드 완료`와 `배치 추론 루프 진입 직전` 사이 시간이 이전보다 더 줄었는지 확인하세요.

In [ ]:
%%writefile scripts/inference.py
import os
import cv2
import math
import copy
import torch
import glob
import shutil
import pickle
import argparse
import numpy as np
import subprocess
import time
from tqdm import tqdm
from omegaconf import OmegaConf
from transformers import WhisperModel
import sys

from musetalk.utils.blending import get_image
from musetalk.utils.face_parsing import FaceParsing
from musetalk.utils.audio_processor import AudioProcessor
from musetalk.utils.utils import get_file_type, get_video_fps, datagen, load_all_model
# musetalk.utils.preprocessing는 import되는 순간 모듈 최상단에서 DWPose 모델 +
# 얼굴 감지 모델을 GPU에 올린다. 캐시 히트 시에는 랜드마크 재계산이 필요 없어서
# 이 두 모델을 아예 쓸 일이 없으므로, 캐시 미스일 때만(실제로 필요할 때만)
# 지연 import한다 - 아래 else 블록 안에서 import.

def fast_check_ffmpeg():
    try:
        subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
        return True
    except:
        return False

_t_start = time.time()
def _t(label):
    print(f"[INF-TIMING] {label}: {round(time.time() - _t_start, 2)}s", flush=True)

@torch.no_grad()
def main(args):
    _t("main() 시작")
    # Configure ffmpeg path
    if not fast_check_ffmpeg():
        print("Adding ffmpeg to PATH")
        # Choose path separator based on operating system
        path_separator = ';' if sys.platform == 'win32' else ':'
        os.environ["PATH"] = f"{args.ffmpeg_path}{path_separator}{os.environ['PATH']}"
        if not fast_check_ffmpeg():
            print("Warning: Unable to find ffmpeg, please ensure ffmpeg is properly installed")

    # Set computing device
    device = torch.device(f"cuda:{args.gpu_id}" if torch.cuda.is_available() else "cpu")
    # Load model weights
    vae, unet, pe = load_all_model(
        unet_model_path=args.unet_model_path,
        vae_type=args.vae_type,
        unet_config=args.unet_config,
        device=device
    )
    timesteps = torch.tensor([0], device=device)

    # Convert models to half precision if float16 is enabled
    if args.use_float16:
        pe = pe.half()
        vae.vae = vae.vae.half()
        unet.model = unet.model.half()

    # Move models to specified device
    pe = pe.to(device)
    vae.vae = vae.vae.to(device)
    unet.model = unet.model.to(device)
    _t("VAE/UNet/PE 로드+GPU 이동 완료")

    # Initialize audio processor and Whisper model
    audio_processor = AudioProcessor(feature_extractor_path=args.whisper_dir)
    weight_dtype = unet.model.dtype
    whisper = WhisperModel.from_pretrained(args.whisper_dir)
    whisper = whisper.to(device=device, dtype=weight_dtype).eval()
    whisper.requires_grad_(False)
    _t("Whisper 로드+GPU 이동 완료")

    # Initialize face parser with configurable parameters based on version
    if args.version == "v15":
        fp = FaceParsing(
            left_cheek_width=args.left_cheek_width,
            right_cheek_width=args.right_cheek_width
        )
    else:  # v1
        fp = FaceParsing()
    _t("FaceParsing 초기화 완료")

    # Load inference configuration
    inference_config = OmegaConf.load(args.inference_config)
    print("Loaded inference config:", inference_config)

    # Process each task
    for task_id in inference_config:
        try:
            # Get task configuration
            video_path = inference_config[task_id]["video_path"]
            audio_path = inference_config[task_id]["audio_path"]
            if "result_name" in inference_config[task_id]:
                args.output_vid_name = inference_config[task_id]["result_name"]

            # Set bbox_shift based on version
            if args.version == "v15":
                bbox_shift = 0  # v15 uses fixed bbox_shift
            else:
                bbox_shift = inference_config[task_id].get("bbox_shift", args.bbox_shift)  # v1 uses config or default

            # Set output paths
            input_basename = os.path.basename(video_path).split('.')[0]
            audio_basename = os.path.basename(audio_path).split('.')[0]
            output_basename = f"{input_basename}_{audio_basename}"

            # Create temporary directories
            temp_dir = os.path.join(args.result_dir, f"{args.version}")
            os.makedirs(temp_dir, exist_ok=True)

            # Set result save paths
            result_img_save_path = os.path.join(temp_dir, output_basename)
            os.makedirs(result_img_save_path, exist_ok=True)

            # Set output video paths
            if args.output_vid_name is None:
                output_vid_name = os.path.join(temp_dir, output_basename + ".mp4")
            else:
                output_vid_name = os.path.join(temp_dir, args.output_vid_name)
            output_vid_name_concat = os.path.join(temp_dir, output_basename + "_concat.mp4")

            # 페르소나(원본 영상) 단위 전체 준비 캐시.
            # frame_list/coord_list/input_latent_list는 오디오(TTS 문장)와 무관하게
            # 같은 원본 영상이면 항상 동일하므로, 요청마다 다시 계산하지 않고 한 번만
            # 계산해서 디스크에 저장해두고 이후 요청은 로드만 한다.
            full_prep_cache_path = os.path.join(
                args.result_dir, "..", f"{input_basename}_full_prep_v{args.version}_em{args.extra_margin}.pt"
            )

            save_dir_full = None
            if os.path.exists(full_prep_cache_path) and args.use_saved_coord:
                _t("전체 준비 캐시 발견 - 프레임 추출/이미지 읽기/랜드마크/VAE latent 인코딩 스킵")
                cache = torch.load(full_prep_cache_path, map_location="cpu")
                frame_list = cache["frame_list"]
                coord_list = cache["coord_list"]
                fps = cache["fps"]
                input_latent_list = [lat.to(device=device, dtype=weight_dtype) for lat in cache["input_latent_list"]]
                _t(f"전체 준비 캐시 로드 완료 ({len(frame_list)}프레임)")
            else:
                # Extract frames from source video
                if get_file_type(video_path) == "video":
                    save_dir_full = os.path.join(temp_dir, input_basename)
                    os.makedirs(save_dir_full, exist_ok=True)
                    cmd = f"ffmpeg -v fatal -i {video_path} -start_number 0 {save_dir_full}/%08d.png"
                    os.system(cmd)
                    input_img_list = sorted(glob.glob(os.path.join(save_dir_full, '*.[jpJP][pnPN]*[gG]')))
                    fps = get_video_fps(video_path)
                    _t(f"원본 영상 프레임 추출 완료 ({len(input_img_list)}장)")
                elif get_file_type(video_path) == "image":
                    input_img_list = [video_path]
                    fps = args.fps
                elif os.path.isdir(video_path):
                    input_img_list = glob.glob(os.path.join(video_path, '*.[jpJP][pnPN]*[gG]'))
                    input_img_list = sorted(input_img_list, key=lambda x: int(os.path.splitext(os.path.basename(x))[0]))
                    fps = args.fps
                else:
                    raise ValueError(f"{video_path} should be a video file, an image file or a directory of images")

                # 캐시 미스일 때만 필요 - DWPose/얼굴 감지 모델을 이 시점에야 GPU에 로드
                from musetalk.utils.preprocessing import get_landmark_and_bbox, coord_placeholder
                _t("DWPose/얼굴 감지 모델 로드 완료 (캐시 미스라 필요)")

                print("Extracting landmarks... time-consuming operation")
                coord_list, frame_list = get_landmark_and_bbox(input_img_list, bbox_shift)
                _t(f"좌표/랜드마크 계산 완료 ({len(frame_list)}프레임)")

                # Process each frame
                input_latent_list = []
                for bbox, frame in zip(coord_list, frame_list):
                    if bbox == coord_placeholder:
                        continue
                    x1, y1, x2, y2 = bbox
                    if args.version == "v15":
                        y2 = y2 + args.extra_margin
                        y2 = min(y2, frame.shape[0])
                    crop_frame = frame[y1:y2, x1:x2]
                    crop_frame = cv2.resize(crop_frame, (256,256), interpolation=cv2.INTER_LANCZOS4)
                    latents = vae.get_latents_for_unet(crop_frame)
                    input_latent_list.append(latents)
                _t(f"프레임별 VAE latent 인코딩 완료 ({len(input_latent_list)}프레임, 1장씩 순차 처리)")

                if args.use_saved_coord:
                    torch.save({
                        "frame_list": frame_list,
                        "coord_list": coord_list,
                        "fps": fps,
                        "input_latent_list": [lat.detach().to("cpu") for lat in input_latent_list],
                    }, full_prep_cache_path)
                    _t("전체 준비 캐시 저장 완료 (다음 요청부터 재사용)")

                if save_dir_full:
                    shutil.rmtree(save_dir_full)

            print(f"Number of frames: {len(frame_list)}")

            # Extract audio features (오디오는 요청마다 다르므로 항상 새로 처리)
            whisper_input_features, librosa_length = audio_processor.get_audio_feature(audio_path)
            whisper_chunks = audio_processor.get_whisper_chunk(
                whisper_input_features,
                device,
                weight_dtype,
                whisper,
                librosa_length,
                fps=fps,
                audio_padding_length_left=args.audio_padding_length_left,
                audio_padding_length_right=args.audio_padding_length_right,
            )
            _t(f"오디오 특징(Whisper) 추출 완료 ({len(whisper_chunks)}청크)")

            # Smooth first and last frames
            frame_list_cycle = frame_list + frame_list[::-1]
            coord_list_cycle = coord_list + coord_list[::-1]
            input_latent_list_cycle = input_latent_list + input_latent_list[::-1]

            # Batch inference (interleaved with blending + disk write, per batch)
            _t("배치 추론 루프 진입 직전")
            print("Starting inference (interleaved infer+blend per batch)")
            video_num = len(whisper_chunks)
            batch_size = args.batch_size
            gen = datagen(
                whisper_chunks=whisper_chunks,
                vae_encode_latents=input_latent_list_cycle,
                batch_size=batch_size,
                delay_frame=0,
                device=device,
            )

            total = int(np.ceil(float(video_num) / batch_size))
            frame_idx = 0  # running position within coord_list_cycle / frame_list_cycle, one per output frame

            # Execute inference; blend + write each batch's frames immediately after that batch decodes
            for i, (whisper_batch, latent_batch) in enumerate(tqdm(gen, total=total)):
                audio_feature_batch = pe(whisper_batch)
                latent_batch = latent_batch.to(dtype=unet.model.dtype)

                pred_latents = unet.model(latent_batch, timesteps, encoder_hidden_states=audio_feature_batch).sample
                recon = vae.decode_latents(pred_latents)

                for res_frame in recon:
                    bbox = coord_list_cycle[frame_idx % (len(coord_list_cycle))]
                    ori_frame = copy.deepcopy(frame_list_cycle[frame_idx % (len(frame_list_cycle))])
                    x1, y1, x2, y2 = bbox
                    if args.version == "v15":
                        y2 = y2 + args.extra_margin
                        y2 = min(y2, ori_frame.shape[0])
                    try:
                        res_frame_resized = cv2.resize(res_frame.astype(np.uint8), (x2 - x1, y2 - y1))
                    except Exception:
                        frame_idx += 1
                        continue

                    # Merge results with version-specific parameters
                    if args.version == "v15":
                        combine_frame = get_image(ori_frame, res_frame_resized, [x1, y1, x2, y2], mode=args.parsing_mode, fp=fp)
                    else:
                        combine_frame = get_image(ori_frame, res_frame_resized, [x1, y1, x2, y2], fp=fp)
                    cv2.imwrite(f"{result_img_save_path}/{str(frame_idx).zfill(8)}.png", combine_frame)
                    frame_idx += 1

            # Save prediction results
            temp_vid_path = f"{temp_dir}/temp_{input_basename}_{audio_basename}.mp4"
            cmd_img2video = f"ffmpeg -y -v warning -r {fps} -f image2 -i {result_img_save_path}/%08d.png -vcodec libx264 -vf format=yuv420p -crf 18 {temp_vid_path}"
            print("Video generation command:", cmd_img2video)
            os.system(cmd_img2video)

            cmd_combine_audio = f"ffmpeg -y -v warning -i {audio_path} -i {temp_vid_path} {output_vid_name}"
            print("Audio combination command:", cmd_combine_audio)
            os.system(cmd_combine_audio)

            # Clean up temporary files
            shutil.rmtree(result_img_save_path)
            os.remove(temp_vid_path)

            print(f"Results saved to {output_vid_name}")
        except Exception as e:
            print("Error occurred during processing:", e)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--ffmpeg_path", type=str, default="./ffmpeg-4.4-amd64-static/", help="Path to ffmpeg executable")
    parser.add_argument("--gpu_id", type=int, default=0, help="GPU ID to use")
    parser.add_argument("--vae_type", type=str, default="sd-vae", help="Type of VAE model")
    parser.add_argument("--unet_config", type=str, default="./models/musetalk/config.json", help="Path to UNet configuration file")
    parser.add_argument("--unet_model_path", type=str, default="./models/musetalkV15/unet.pth", help="Path to UNet model weights")
    parser.add_argument("--whisper_dir", type=str, default="./models/whisper", help="Directory containing Whisper model")
    parser.add_argument("--inference_config", type=str, default="configs/inference/test_img.yaml", help="Path to inference configuration file")
    parser.add_argument("--bbox_shift", type=int, default=0, help="Bounding box shift value")
    parser.add_argument("--result_dir", default='./results', help="Directory for output results")
    parser.add_argument("--extra_margin", type=int, default=10, help="Extra margin for face cropping")
    parser.add_argument("--fps", type=int, default=25, help="Video frames per second")
    parser.add_argument("--audio_padding_length_left", type=int, default=2, help="Left padding length for audio")
    parser.add_argument("--audio_padding_length_right", type=int, default=2, help="Right padding length for audio")
    parser.add_argument("--batch_size", type=int, default=8, help="Batch size for inference")
    parser.add_argument("--output_vid_name", type=str, default=None, help="Name of output video file")
    parser.add_argument("--use_saved_coord", action="store_true", help='페르소나(원본 영상) 단위 전체 준비 캐시(frame/coord/latent)를 사용/저장')
    parser.add_argument("--saved_coord", action="store_true", help='(미사용, 이전 버전과의 CLI 호환용으로만 남김)')
    parser.add_argument("--use_float16", action="store_true", help="Use float16 for faster inference")
    parser.add_argument("--parsing_mode", default='jaw', help="Face blending parsing mode")
    parser.add_argument("--left_cheek_width", type=int, default=90, help="Width of left cheek region")
    parser.add_argument("--right_cheek_width", type=int, default=90, help="Width of right cheek region")
    parser.add_argument("--version", type=str, default="v15", choices=["v1", "v15"], help="Model version to use")
    args = parser.parse_args()
    main(args)


## 7. 페르소나 A 영상 업로드 + TTS 생성 (OpenAI TTS)

공식 샘플 대신 실제 위원 A 아바타 영상으로 테스트합니다. 먼저 Colab 파일 탐색기에서 `MuseTalk` 폴더 안에 `data/persona_a/` 폴더를 만들고, 로컬의 `ai/media/assets/persona_a/avata_rf.mp4`를 그 안에 업로드하세요 (파일이 작아 Drive 없이 직접 업로드 가능).

TTS 엔진은 `ai/media/tts/tts_engine_decision.md`에서 **OpenAI TTS(`gpt-4o-mini-tts`)로 확정**했습니다 (edge-tts보다 한국어 음질이 낫다고 판단, 프로젝트가 이미 OpenAI API를 LLM 파이프라인에 쓰고 있어 별도 계정 불필요). 보이스/톤은 `ai/media/tts/voice_map.json`을 그대로 따릅니다.

**API 키 설정**: Colab 왼쪽 사이드바의 열쇠(🔑) 아이콘 → 보안 비밀(Secrets) → `OPENAI_API_KEY` 이름으로 키 추가 (노트북에 직접 하드코딩하지 않기 위함).

In [ ]:
from google.colab import userdata
import os
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
pip install openai

In [ ]:
%%bash
# OPENAI_API_KEY는 앞 셀에서 os.environ에 설정해둔 걸 그대로 상속받음
source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk
mkdir -p data/audio  # 새 세션에서는 이 폴더가 없어서 mp3 저장이 실패하므로 매번 보장
python << 'PYEOF'
from openai import OpenAI

client = OpenAI()

# voice_map.json (business_strategy / persona_a) 그대로 반영
text = "안녕하세요. 사업 전략 관점에서 이번 계획서를 검토했습니다. 시장 진입 전략은 탄탄하지만, 수익 모델 부분은 조금 더 구체화가 필요해 보입니다."
voice = "nova"
instructions = "천천히 자연스럽게, 아나운서처럼 발표하듯이, 여성스러우면서 자연스러운 톤으로 말하세요."

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice=voice,
    input=text,
    instructions=instructions,
) as response:
    response.stream_to_file("data/audio/persona_a_tts.mp3")

print("TTS 생성 완료")
PYEOF

# MuseTalk 샘플들이 wav를 쓰므로 동일하게 wav로 변환
ffmpeg -y -v error -i data/audio/persona_a_tts.mp3 data/audio/persona_a_tts.wav

In [ ]:
# 생성된 TTS를 바로 들어보기 (립싱크 GPU 단계로 넘어가기 전에 음성 자체부터 확인)
from IPython.display import Audio, display
display(Audio("data/audio/persona_a_tts.wav"))

## 8. TTS 구간 립싱크 추론

TTS 오디오(패딩 없음)로 립싱크 영상을 생성합니다. 10초 단위로 맞추는 건 이 단계가 아니라 10번 단계(무음 tail 이어붙이기)에서 처리합니다 — 오디오를 무음으로 패딩해서 한 번에 생성하면 MuseTalk이 무음 구간에서도 입을 움직이는 문제가 있었기 때문입니다 (아래 10번 참고).

In [ ]:
%%writefile configs/inference/persona_a_tts.yaml
task_persona_a_tts:
  video_path: "data/persona_a/avata_rf.mp4"
  audio_path: "data/audio/persona_a_tts.wav"

In [ ]:
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
MPLBACKEND=Agg python -m scripts.inference \
  --inference_config configs/inference/persona_a_tts.yaml \
  --result_dir results/persona_a_tts \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 \
  --extra_margin 15 \
  --audio_padding_length_right 0

## 9. 입 주변 색보정 + 우글거림(jitter) 완화

`extra_margin=15`만으로 부족할 때 사용합니다. MuseTalk이 입 영역을 VAE로 새로 합성하는 과정에서 원본 영상과 미묘하게 색감이 달라지는 문제가 있어, **원본 영상의 입 주변 색을 기준으로 보정**하고, 동시에 이웃 프레임과 블렌딩해 시간축 우글거림도 완화합니다. 색보정은 입 주변에만 부드럽게(페더링) 적용해 나머지 얼굴/배경은 그대로 유지합니다. MuseTalk이 인코딩 후 중간 PNG 프레임을 삭제하므로, 8번에서 만들어진 최종 mp4에서 프레임을 다시 추출한 뒤 처리합니다. (8번 단계의 립싱크 구간에만 적용 — 10번의 무음 tail은 원본 영상 그대로라 필요 없습니다.)

최종 mp4에서 프레임을 추출해 스무딩을 적용합니다. `!`가 붙지 않은 일반 코드 셀이라 `musetalk` 가상환경이 아니라 Colab 기본 커널(Python 3.12)에서 실행되는데, numpy/Pillow/ffmpeg 모두 기본적으로 사용 가능합니다.

In [ ]:
import subprocess, glob, os
import numpy as np
from PIL import Image

SRC = "results/persona_a_tts/v15/avata_rf_persona_a_tts.mp4"
REF = "data/persona_a/avata_rf.mp4"
frame_dir = "results/persona_a_tts/v15/_frames_smooth"
os.makedirs(frame_dir, exist_ok=True)

# 실제 fps를 하드코딩하지 않고 파일에서 읽어옴 (페르소나 A 영상은 24fps라 25로 고정하면 어긋남)
fps_str = subprocess.check_output(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=r_frame_rate", "-of", "csv=p=0", SRC]
).decode().strip()
_num, _den = fps_str.split('/')
FPS = round(float(_num) / float(_den), 3)
print("감지된 FPS:", FPS)

subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", SRC, f"{frame_dir}/%08d.png"], check=True)
files = sorted(glob.glob(os.path.join(frame_dir, "*.png")))
print(len(files), "frames extracted")

# 원본 영상 기준 프레임에서 입 주변 색 평균을 구해 보정 기준으로 사용
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", REF, "-vframes", "1", "ref_frame.png"], check=True)
ref_img = np.asarray(Image.open("ref_frame.png").convert("RGB"), dtype=np.float32)

# 입 주변 크롭 영역 (x, y, w, h) - 720x1280 프레임 기준, 필요시 좌표 조정
mx, my, mw, mh = 260, 430, 200, 120
ref_mouth_mean = ref_img[my:my + mh, mx:mx + mw].reshape(-1, 3).mean(axis=0)

# 크롭 박스 중심으로 부드럽게 퍼지는 타원형 마스크 (사각형 경계가 티 안 나게 페더링)
H, W = ref_img.shape[:2]
yy, xx = np.mgrid[0:H, 0:W]
cy, cx = my + mh / 2, mx + mw / 2
ry, rx = mh * 0.9, mw * 0.9
dist = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2
mask = np.clip(1.5 - dist, 0, 1)[..., None]

w_prev, w_curr, w_next = 0.15, 0.70, 0.15
frames = [np.asarray(Image.open(f).convert("RGB"), dtype=np.float32) for f in files]

corrected = []
for frame in frames:
    frame_mouth_mean = frame[my:my + mh, mx:mx + mw].reshape(-1, 3).mean(axis=0)
    offset = ref_mouth_mean - frame_mouth_mean
    corrected.append(np.clip(frame + offset[None, None, :] * mask, 0, 255))

for i, f in enumerate(files):
    prev_f = corrected[i - 1] if i > 0 else corrected[i]
    next_f = corrected[i + 1] if i < len(corrected) - 1 else corrected[i]
    blended = w_prev * prev_f + w_curr * corrected[i] + w_next * next_f
    Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8)).save(f)

print("색보정 + 스무딩 완료:", len(files), "frames")

In [ ]:
temp_video = "results/persona_a_tts/v15/temp_smoothed.mp4"
final_video = "results/persona_a_tts/v15/avata_rf_persona_a_tts_smoothed.mp4"
print(temp_video)
print(final_video)

In [ ]:
subprocess.run(["ffmpeg", "-y", "-v", "warning", "-r", str(FPS), "-f", "image2", "-i", f"{frame_dir}/%08d.png",
                 "-vcodec", "libx264", "-vf", "format=yuv420p", "-crf", "18", temp_video], check=True)
subprocess.run(["ffmpeg", "-y", "-v", "warning", "-i", temp_video, "-i", SRC,
                 "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-c:a", "copy", "-shortest", final_video], check=True)
print("스무딩 적용된 영상:", final_video)

## 10. 무음 tail 이어붙여서 10초 단위로 맞추기

오디오를 무음으로 패딩해서 MuseTalk에 한 번에 넣는 방식(1차 시도)은 실패했습니다 — 무음 구간에서도 입을 계속 움직였습니다.

`frame_list_cycle` 위치를 정확히 계산해서 자세를 이어붙이는 방식(2~3차 시도)도 부족했습니다 — 자세(몸/머리)는 이어져도 **입모양 자체가 여전히 다를 수 있습니다.** MuseTalk이 생성한 마지막 입모양과, 원본 루프의 그 시점 입모양이 우연히 비슷할 거라는 보장이 없기 때문입니다.

**4차 시도(이번)**: 립싱크 마지막 프레임을 뽑아서, 원본 루프 241프레임 전체 중 **입 주변이 픽셀 단위로 가장 비슷한 프레임**을 찾아 그 지점부터 tail을 시작합니다. "정확한 재생 위치"보다 "입모양이 가장 비슷한 위치"를 우선합니다.

In [ ]:
import subprocess, math, os, glob, re
import numpy as np
from PIL import Image

def run(cmd):
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True, stdin=subprocess.DEVNULL)

TALK = "results/persona_a_tts/v15/avata_rf_persona_a_tts_smoothed.mp4"
if not os.path.exists(TALK):
    TALK = "results/persona_a_tts/v15/avata_rf_persona_a_tts.mp4"  # 9번(스무딩) 건너뛴 경우
print("사용할 립싱크 클립:", TALK)

REF = "data/persona_a/avata_rf.mp4"
TTS_WAV = "data/audio/persona_a_tts.wav"

_fps_str = subprocess.check_output(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=r_frame_rate", "-of", "csv=p=0", TALK]
).decode().strip()
_num, _den = _fps_str.split('/')
FPS = round(float(_num) / float(_den), 3)
print("감지된 FPS:", FPS)

# --- 실제 발화 종료 시점(+여유)에서 자르기, 끝에 오디오 페이드아웃 ---
silence_log = subprocess.run(
    ["ffmpeg", "-i", TTS_WAV, "-af", "silencedetect=noise=-30dB:d=0.15", "-f", "null", "-"],
    stderr=subprocess.PIPE, check=True
).stderr.decode()
starts = [float(x) for x in re.findall(r"silence_start:\s*([\d.]+)", silence_log)]
full_dur = float(subprocess.check_output(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", TTS_WAV]
).decode().strip())
speech_end_raw = starts[-1] if starts else full_dur
BUFFER = 0.15
talk_dur = min(round(speech_end_raw + BUFFER, 2), full_dur)
print(f"발화 종료(감지): {speech_end_raw:.2f}s -> 여유 포함 자르는 지점: {talk_dur:.2f}s")

target = math.ceil(talk_dur / 10.0) * 10
tail = round(target - talk_dur, 2)
print(f"목표: {target}s (tail {tail:.2f}s)")

FINAL = "results/persona_a_final.mp4"

# --- 1. TALK을 자르고 동일 스펙으로 통일, 끝부분 오디오 페이드아웃 ---
fade_start = max(0, round(talk_dur - 0.15, 2))
run(["ffmpeg", "-y", "-v", "error", "-i", TALK, "-t", str(talk_dur),
     "-r", str(FPS), "-vf", "scale=720:1280,format=yuv420p",
     "-af", f"afade=t=out:st={fade_start}:d=0.15",
     "-c:v", "libx264", "-crf", "18",
     "-ar", "44100", "-ac", "2", "-c:a", "aac",
     "norm_talk.mp4"])
print("1/6 norm_talk.mp4 완료")

# --- 2. norm_talk.mp4의 마지막 프레임을 뽑아서, 원본 루프 프레임 중 입모양이 가장 비슷한 프레임을 탐색 ---
run(["ffmpeg", "-y", "-v", "error", "-sseof", "-0.1", "-i", "norm_talk.mp4", "-vframes", "1", "talk_last_frame.png"])

orig_frame_dir = "data/persona_a/_orig_frames"
os.makedirs(orig_frame_dir, exist_ok=True)
if not glob.glob(os.path.join(orig_frame_dir, "*.png")):
    run(["ffmpeg", "-y", "-v", "error", "-i", REF, f"{orig_frame_dir}/%08d.png"])
orig_files = sorted(glob.glob(os.path.join(orig_frame_dir, "*.png")))
N = len(orig_files)
print("원본 프레임 수 N =", N)

mx, my, mw, mh = 260, 430, 200, 120
last_mouth = np.asarray(Image.open("talk_last_frame.png").convert("RGB").resize((720, 1280)), dtype=np.float32)[my:my + mh, mx:mx + mw]

best_idx, best_dist = 0, None
for idx, fpath in enumerate(orig_files):
    cand = np.asarray(Image.open(fpath).convert("RGB"), dtype=np.float32)[my:my + mh, mx:mx + mw]
    dist = float(np.mean((cand - last_mouth) ** 2))
    if best_dist is None or dist < best_dist:
        best_dist, best_idx = dist, idx
print(f"가장 입모양이 비슷한 원본 프레임: {best_idx}번 (거리 {best_dist:.1f})")

# --- 3. 그 지점부터 정방향+역방향 핑퐁 규칙으로 이어서 tail 프레임 구성 ---
cycle_len = 2 * N
def cycle_index(j):
    p = j % cycle_len
    return p if p < N else (2 * N - 1 - p)

tail_frames_needed = int(round(tail * FPS))
tail_indices = [cycle_index(best_idx + k) for k in range(tail_frames_needed)]
print(f"tail 프레임 {tail_frames_needed}개, {best_idx}번부터 이어짐")

tail_frame_dir = "results/persona_a_tts/v15/_tail_frames"
os.makedirs(tail_frame_dir, exist_ok=True)
for k, idx in enumerate(tail_indices):
    Image.open(orig_files[idx]).save(os.path.join(tail_frame_dir, f"{k+1:08d}.png"))
print("2/6 tail 프레임 생성 완료")

run(["ffmpeg", "-y", "-v", "error", "-r", str(FPS), "-f", "image2",
     "-i", f"{tail_frame_dir}/%08d.png",
     "-vf", "scale=720:1280,format=yuv420p", "-c:v", "libx264", "-crf", "18",
     "norm_tail_video.mp4"])
print("3/6 norm_tail_video.mp4 완료")

# --- 4. 무음 오디오 생성 + tail 영상과 결합 ---
run(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo",
     "-t", str(tail), "-c:a", "aac", "norm_tail_audio.aac"])
print("4/6 norm_tail_audio.aac 완료")

run(["ffmpeg", "-y", "-v", "error", "-i", "norm_tail_video.mp4", "-i", "norm_tail_audio.aac",
     "-c:v", "copy", "-c:a", "copy", "-shortest", "norm_tail.mp4"])
print("5/6 norm_tail.mp4 완료")

# --- 5. 크로스페이드(0.3s)로 이어붙이기 ---
xfade_dur = min(0.3, round(talk_dur / 2, 2))
offset = round(talk_dur - xfade_dur, 2)
run(["ffmpeg", "-y", "-v", "error", "-i", "norm_talk.mp4", "-i", "norm_tail.mp4",
     "-filter_complex",
     f"[0:v][1:v]xfade=transition=fade:duration={xfade_dur}:offset={offset}[v];"
     f"[0:a][1:a]acrossfade=d={xfade_dur}[a]",
     "-map", "[v]", "-map", "[a]",
     "-c:v", "libx264", "-pix_fmt", "yuv420p", "-profile:v", "high", "-level", "4.0",
     "-c:a", "aac", "-ar", "44100", "-ac", "2",
     "-movflags", "+faststart",
     FINAL])
print("6/6 완료:", FINAL)

## 11. FastAPI 서버 작성 — 모델 상주 구조 + MSE 연속 스트리밍

**구조 변경 1(모델 상주)**: 처음에는 요청마다 `python -m scripts.inference`를 새 프로세스로 띄우는 방식이었는데, 실측 결과 이 방식은 모델 로드(~12초)를 매 요청마다 반복하는 게 가장 큰 병목이었습니다. 그래서 서버가 **시작될 때 UNet/VAE/PE/Whisper/FaceParsing을 딱 한 번만 GPU에 로드**해서 전역으로 들고 있고, 요청은 subprocess 대신 **이미 로드된 모델을 함수로 직접 호출**하도록 바꿨습니다. 페르소나(원본 영상)별 준비물(frame/coord/latent)도 메모리에 캐싱해서, 서버가 살아있는 한 같은 페르소나는 다시 계산하지 않습니다. (실측: 첫 청크까지 74~78초 → 6.64초. 자세한 진단 과정은 `ai/media/musetalk/streaming_optimization.md` 참고)

**구조 변경 2(MSE 연속 스트리밍)**: 기존에는 프레임을 1초 분량씩 잘라서 **독립된 mp4 파일 여러 개**를 만들어 보냈는데, 프론트엔드에서 `<video>.src`를 매번 갈아 끼우는 방식이라 청크 경계마다 디코더가 재시작되면서 끊겼습니다. 이번엔 프레임이 나오는 대로 **하나의 연속된 fragmented MP4(fMP4) muxing 프로세스**에 흘려보내고, 그 바이트를 그대로 WebSocket으로 전달합니다. 프론트엔드(`web_test_stream_mse.html`)는 `MediaSource`의 `SourceBuffer.appendBuffer()`로 이어붙여서 재생하므로 끊김이 없습니다.

**서버가 뜨는 데 이전보다 오래 걸립니다**(모델 로드 + 페르소나 워밍업, 최초 캐시가 없으면 최대 1분 가까이). 12번 단계의 헬스체크 대기시간도 이에 맞춰 늘려뒀습니다.

품질 후처리(색보정/스무딩/10초 패딩)는 여전히 빼고 TTS + 립싱크만 넣은 상태입니다. 요청 필드는 `contracts/`의 `mediaLine` 스키마(`speaker_id`, `speaker_name`, `order`, `text`, `emotion`)를 참고해 최소한으로 맞췄습니다. `speaker_id`로 어떤 페르소나 영상+보이스를 쓸지 결정합니다(`ai/media/tts/voice_map.json` 반영, 지금은 `persona_a`만 존재). TTS는 OpenAI TTS(`gpt-4o-mini-tts`)를 사용합니다 — `OPENAI_API_KEY`는 7번 단계에서 이미 환경변수로 설정해뒀어야 합니다.

**테스트는 `web_test_stream.html`이 아니라 `web_test_stream_mse.html`을 사용하세요** — 프로토콜(청크 JSON 메시지 없이 순수 바이너리 스트림)이 바뀌어서 예전 페이지로는 안 됩니다.

In [ ]:
!source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && \
pip install fastapi "uvicorn[standard]"

In [ ]:
%%writefile app.py
from fastapi import FastAPI, HTTPException, WebSocket
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
import asyncio, glob, os, queue, shutil, subprocess, threading, time
import cv2
import numpy as np
import torch
from tqdm import tqdm
from transformers import WhisperModel

from musetalk.utils.blending import get_image_blending, get_image_prepare_material
from musetalk.utils.face_parsing import FaceParsing
from musetalk.utils.audio_processor import AudioProcessor
from musetalk.utils.utils import get_video_fps, datagen, load_all_model

app = FastAPI()

# 프로토타입 단계라 전체 허용 - 실제 서비스 단계에서는 프론트 도메인만 허용하도록 좁혀야 함
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

client = OpenAI()

# 프론트엔드가 대기(idle) 상태일 때 재생할 원본 루프 영상을 직접 내려받는 용도.
# 예: {서버주소}/media/persona_a/avata_rf.mp4 (web_test_chat.html에서 사용)
app.mount("/media", StaticFiles(directory="data"), name="media")

# GPU는 공유 자원이라 워커 1개로 요청을 직렬화 (동시 요청이 GPU에서 서로 뒤섞이는 것 방지)
EXECUTOR = ThreadPoolExecutor(max_workers=1)

# ai/media/tts/voice_map.json 반영
PERSONA_MAP = {
    "business_strategy": {
        "video_path": "data/persona_a/avata_rf.mp4",
        "voice": "nova",
        "instructions": "천천히 자연스럽게, 아나운서처럼 발표하듯이, 여성스러우면서 자연스러운 톤으로 말하세요.",
    },
    # persona_b(cedar)는 영상 준비되면 추가
}

VERSION = "v15"
EXTRA_MARGIN = 15
UNET_MODEL_PATH = "models/musetalkV15/unet.pth"
UNET_CONFIG = "models/musetalkV15/musetalk.json"
WHISPER_DIR = "models/whisper"
VAE_TYPE = "sd-vae"
PARSING_MODE = "jaw"
LEFT_CHEEK_WIDTH = 90
RIGHT_CHEEK_WIDTH = 90
AUDIO_PAD_LEFT = 2
AUDIO_PAD_RIGHT = 0
# 페이서가 목표로 하는 전달 배속. 1.0이면 재생 속도와 정확히 같아서 생성 속도가
# 실시간보다 빨라도(1.6~1.8배속 실측) 그 여유분이 버퍼로 못 쌓이고 그대로 소진돼버림
# (streaming_optimization.md 시도 9 참고). 재생 속도보다 살짝 빠르게 전달해야
# 버퍼가 점점 쌓여서 이후 멈춤 없이 재생됨.
PACE_SPEED_MULTIPLIER = 1.5
BATCH_SIZE = 16

# ---- 모델 상주 상태: 서버 시작 시 한 번만 채워지고, 이후 모든 요청이 재사용 ----
_device = None
_vae = None
_unet = None
_pe = None
_whisper = None
_audio_processor = None
_fp = None
_weight_dtype = None
_timesteps = None
_persona_cache: dict[str, dict] = {}  # video_path -> {frame_list, coord_list, fps, input_latent_list}


def _mt(label: str, t0: float):
    print(f"[ML-TIMING] {label}: {round(time.time() - t0, 2)}s", flush=True)


def _blend_fast(image, face, face_box, mask_array, crop_box):
    """get_image_blending()과 결과는 동일하지만 PIL을 전혀 안 쓰고 crop_box 영역만
    numpy로 직접 합성한다. get_image_blending()은 매 프레임 720x1280 원본 전체를
    PIL Image로 변환했다가 다시 numpy로 되돌리는데, 실제로 바뀌는 건 crop_box(얼굴
    주변 작은 영역)뿐이라 이 변환이 프레임당 CPU 비용의 상당 부분을 차지했다.
    crop_box가 프레임 경계를 벗어나는 예외적인 경우만 안전하게 기존 함수로 폴백."""
    x, y, x1, y1 = face_box
    x_s, y_s, x_e, y_e = crop_box
    h, w = image.shape[:2]
    if x_s < 0 or y_s < 0 or x_e > w or y_e > h:
        return get_image_blending(image, face, face_box, mask_array, crop_box)

    out = image.copy()
    region = out[y_s:y_e, x_s:x_e]
    patch = region.copy()
    patch[y - y_s:y1 - y_s, x - x_s:x1 - x_s] = face

    mask = mask_array.astype(np.float32)
    if mask.ndim == 2:
        mask = mask[:, :, np.newaxis]
    mask /= 255.0

    region[:] = (patch.astype(np.float32) * mask + region.astype(np.float32) * (1.0 - mask)).astype(np.uint8)
    return out


def load_models():
    """서버 시작 시 단 한 번 호출. UNet/VAE/PE/Whisper/FaceParsing을 GPU에 로드해 전역에 상주시킨다."""
    global _device, _vae, _unet, _pe, _whisper, _audio_processor, _fp, _weight_dtype, _timesteps
    t0 = time.time()
    _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    _vae, _unet, _pe = load_all_model(
        unet_model_path=UNET_MODEL_PATH,
        vae_type=VAE_TYPE,
        unet_config=UNET_CONFIG,
        device=_device,
    )
    _timesteps = torch.tensor([0], device=_device)
    _pe = _pe.to(_device)
    _vae.vae = _vae.vae.to(_device)
    _unet.model = _unet.model.to(_device)
    _mt("VAE/UNet/PE 로드 완료", t0)

    _audio_processor = AudioProcessor(feature_extractor_path=WHISPER_DIR)
    _weight_dtype = _unet.model.dtype
    _whisper = WhisperModel.from_pretrained(WHISPER_DIR).to(device=_device, dtype=_weight_dtype).eval()
    _whisper.requires_grad_(False)
    _mt("Whisper 로드 완료", t0)

    _fp = FaceParsing(left_cheek_width=LEFT_CHEEK_WIDTH, right_cheek_width=RIGHT_CHEEK_WIDTH)
    _mt("FaceParsing 초기화 완료", t0)


def _prepare_persona(video_path: str) -> dict:
    """페르소나(원본 영상) 단위 준비물(frame/coord/latent)을 메모리 캐시로 관리.
    프로세스가 살아있는 한 같은 영상은 다시 계산하지 않는다. 프로세스 재시작 시에는
    디스크 캐시(results/*_full_prep_*.pt)가 있으면 그걸 불러오고, 그마저 없으면 새로 계산한다."""
    if video_path in _persona_cache:
        return _persona_cache[video_path]

    t0 = time.time()
    input_basename = os.path.splitext(os.path.basename(video_path))[0]
    os.makedirs("results", exist_ok=True)
    # full_prep2: 마스크/crop_box 캐싱이 추가된 스키마 (이전 full_prep 캐시와 호환 안 됨)
    disk_cache_path = f"results/{input_basename}_full_prep2_v{VERSION}_em{EXTRA_MARGIN}.pt"

    if os.path.exists(disk_cache_path):
        cache = torch.load(disk_cache_path, map_location="cpu")
        data = {
            "frame_list": cache["frame_list"],
            "coord_list": cache["coord_list"],
            "fps": cache["fps"],
            "input_latent_list": [lat.to(device=_device, dtype=_weight_dtype) for lat in cache["input_latent_list"]],
            "mask_list": cache["mask_list"],
            "crop_box_list": cache["crop_box_list"],
        }
        _mt(f"페르소나 준비 - 디스크 캐시 로드 완료 ({video_path}, {len(data['frame_list'])}프레임)", t0)
        _persona_cache[video_path] = data
        return data

    # 디스크 캐시도 없으면 최초 1회 처음부터 계산 (느림 - DWPose/얼굴 감지 모델은 이때만 필요해서 지연 import)
    from musetalk.utils.preprocessing import get_landmark_and_bbox, coord_placeholder

    save_dir_full = f"results/_prep_{input_basename}"
    os.makedirs(save_dir_full, exist_ok=True)
    subprocess.run(
        f"ffmpeg -v fatal -y -i {video_path} -start_number 0 {save_dir_full}/%08d.png",
        shell=True, check=True,
    )
    input_img_list = sorted(glob.glob(os.path.join(save_dir_full, "*.png")))
    fps = get_video_fps(video_path)
    _mt(f"페르소나 준비 - 프레임 추출 완료 ({len(input_img_list)}장)", t0)

    coord_list, frame_list = get_landmark_and_bbox(input_img_list, 0)
    _mt("페르소나 준비 - 좌표/랜드마크 계산 완료", t0)

    # 얼굴 파싱(FaceParsing) 신경망 forward pass는 프레임 위치당 한 번만 필요 - 마스크를
    # 여기서 미리 계산해서 캐싱해두면, 실제 생성 시(run_inference_sync)에는 신경망을
    # 다시 안 돌리고 가벼운 get_image_blending()만 호출하면 된다.
    input_latent_list = []
    mask_list = []
    crop_box_list = []
    with torch.no_grad():
        for bbox, frame in zip(coord_list, frame_list):
            if bbox == coord_placeholder:
                continue
            x1, y1, x2, y2 = bbox
            y2 = min(y2 + EXTRA_MARGIN, frame.shape[0])
            crop_frame = cv2.resize(frame[y1:y2, x1:x2], (256, 256), interpolation=cv2.INTER_LANCZOS4)
            input_latent_list.append(_vae.get_latents_for_unet(crop_frame))
            mask_array, crop_box = get_image_prepare_material(frame, [x1, y1, x2, y2], mode=PARSING_MODE, fp=_fp)
            mask_list.append(mask_array)
            crop_box_list.append(crop_box)
    _mt(f"페르소나 준비 - VAE latent+마스크 계산 완료 ({len(input_latent_list)}프레임)", t0)

    shutil.rmtree(save_dir_full)

    torch.save({
        "frame_list": frame_list,
        "coord_list": coord_list,
        "fps": fps,
        "input_latent_list": [lat.detach().to("cpu") for lat in input_latent_list],
        "mask_list": mask_list,
        "crop_box_list": crop_box_list,
    }, disk_cache_path)
    _mt("페르소나 준비 - 디스크 캐시 저장 완료", t0)

    data = {
        "frame_list": frame_list, "coord_list": coord_list, "fps": fps,
        "input_latent_list": input_latent_list, "mask_list": mask_list, "crop_box_list": crop_box_list,
    }
    _persona_cache[video_path] = data
    return data


def run_inference_sync(video_path: str, audio_path: str, result_img_save_path: str | None = None,
                        frame_callback=None) -> float:
    """이미 로드된 모델 + 페르소나 캐시로 프레임을 생성한다. 블로킹 함수이므로 반드시
    스레드풀(EXECUTOR)에서 호출할 것. 반환값은 fps.

    - result_img_save_path가 주어지면 각 프레임을 그 디렉터리에 PNG로 저장 (/generate용).
    - frame_callback이 주어지면 프레임이 생기는 즉시(같은 스레드 안에서, 압축 없는 raw
      BGR bytes로) 호출한다 (/generate-stream이 ffmpeg stdin에 바로 흘려보내는 용도 -
      어차피 ffmpeg가 바로 다시 인코딩할 거라 중간에 PNG로 압축했다 풀 이유가 없음.
      폴링 없이 프레임 생성과 같은 스레드에서 처리해야 asyncio 이벤트 루프와의 GIL
      경합으로 인한 지연이 없다).
    """
    persona = _prepare_persona(video_path)
    frame_list = persona["frame_list"]
    coord_list = persona["coord_list"]
    fps = persona["fps"]
    input_latent_list = persona["input_latent_list"]
    mask_list = persona["mask_list"]
    crop_box_list = persona["crop_box_list"]

    whisper_input_features, librosa_length = _audio_processor.get_audio_feature(audio_path)
    whisper_chunks = _audio_processor.get_whisper_chunk(
        whisper_input_features,
        _device,
        _weight_dtype,
        _whisper,
        librosa_length,
        fps=fps,
        audio_padding_length_left=AUDIO_PAD_LEFT,
        audio_padding_length_right=AUDIO_PAD_RIGHT,
    )

    frame_list_cycle = frame_list + frame_list[::-1]
    coord_list_cycle = coord_list + coord_list[::-1]
    input_latent_list_cycle = input_latent_list + input_latent_list[::-1]
    mask_list_cycle = mask_list + mask_list[::-1]
    crop_box_list_cycle = crop_box_list + crop_box_list[::-1]

    video_num = len(whisper_chunks)
    gen = datagen(
        whisper_chunks=whisper_chunks,
        vae_encode_latents=input_latent_list_cycle,
        batch_size=BATCH_SIZE,
        delay_frame=0,
        device=_device,
    )
    total = int(np.ceil(float(video_num) / BATCH_SIZE))
    frame_idx = 0

    with torch.no_grad():
        for whisper_batch, latent_batch in tqdm(gen, total=total):
            audio_feature_batch = _pe(whisper_batch)
            latent_batch = latent_batch.to(dtype=_unet.model.dtype)
            pred_latents = _unet.model(latent_batch, _timesteps, encoder_hidden_states=audio_feature_batch).sample
            recon = _vae.decode_latents(pred_latents)

            for res_frame in recon:
                bbox = coord_list_cycle[frame_idx % len(coord_list_cycle)]
                ori_frame = frame_list_cycle[frame_idx % len(frame_list_cycle)]  # _blend_fast가 내부에서 복사함
                x1, y1, x2, y2 = bbox
                y2 = min(y2 + EXTRA_MARGIN, ori_frame.shape[0])
                try:
                    res_frame_resized = cv2.resize(res_frame.astype(np.uint8), (x2 - x1, y2 - y1))
                except Exception:
                    frame_idx += 1
                    continue
                mask_array = mask_list_cycle[frame_idx % len(mask_list_cycle)]
                crop_box = crop_box_list_cycle[frame_idx % len(crop_box_list_cycle)]
                combine_frame = _blend_fast(ori_frame, res_frame_resized, [x1, y1, x2, y2], mask_array, crop_box)
                if result_img_save_path:
                    cv2.imwrite(f"{result_img_save_path}/{str(frame_idx).zfill(8)}.png", combine_frame)
                if frame_callback:
                    # PNG 압축 없이 raw BGR bytes 그대로 - 어차피 ffmpeg가 바로 인코딩하므로
                    # 압축했다 푸는 왕복이 순수 낭비
                    frame_callback(np.ascontiguousarray(combine_frame).tobytes())
                frame_idx += 1

    # 발화가 끝나자마자 뚝 끊기지 않고 원본 루프 영상으로 자연스럽게 이어지도록,
    # 남은 구간을 무음 tail(원본 프레임 그대로, 립싱크 없음)로 채워서 전체 길이를
    # 루프 자체 길이(loop_len)의 배수로 맞춘다. 예: 루프가 241프레임이고 발화가
    # 320프레임이면 목표는 482프레임(241의 배수 중 320 이상인 최소값), tail은
    # 162프레임 - loop_resume_offset(=320 mod 241=79)부터 루프 끝까지.
    # tail은 MuseTalk 추론(UNet/VAE) 없이 원본 프레임을 그대로 흘려보내므로 추가
    # GPU 비용이 거의 없다. ping-pong(frame_list_cycle)이 아니라 "루프를 forward로만
    # 감았을 때"의 위치를 기준으로 잡아야, tail이 정확히 루프의 마지막 프레임에서
    # 끝나서 프론트엔드의 단순 반복 재생(<video loop>)으로 넘어갈 때 끊기지 않는다.
    # (발화 중 배경 자세 선택은 기존 ping-pong을 그대로 두므로, 발화->tail 경계에서
    # 아주 미세한 자세 튐이 있을 수 있음 - 알려진 한계, streaming_optimization.md 참고)
    loop_len = len(frame_list)
    target_frames = int(np.ceil(video_num / loop_len)) * loop_len
    tail_frames = target_frames - video_num
    if tail_frames > 0:
        loop_resume_offset = video_num % loop_len
        for tail_frame in frame_list[loop_resume_offset:]:
            if result_img_save_path:
                cv2.imwrite(f"{result_img_save_path}/{str(frame_idx).zfill(8)}.png", tail_frame)
            if frame_callback:
                frame_callback(np.ascontiguousarray(tail_frame).tobytes())
            frame_idx += 1

    return fps


def _ensure_idle_loop_video(video_path: str) -> str:
    """프론트엔드가 대기 상태일 때 트는 루프 영상은 원본 그대로 서빙하면 안 된다.
    원본(720x1280, 약 1.46Mbps)이 스트리밍용으로 튜닝한 값(480x854/crf28, 약
    0.5~1Mbps)보다 비트레이트가 높아서, 무료 Cloudflare Quick Tunnel 대역폭에서
    실측으로 버퍼링이 심하게 발생했다(MuseTalk 스트림과 무관하게 정적 파일
    다운로드만으로도). 스트리밍과 동일한 해상도/crf로 한 번 압축해두고 그
    압축본을 서빙한다 - 화질도 스트림과 일치해서 대기->발화 전환 시 화질이
    갑자기 바뀌는 위화감도 없앤다. 결과는 디스크에 캐싱해 재계산하지 않는다."""
    base, ext = os.path.splitext(video_path)
    idle_path = f"{base}_idle480{ext}"
    if os.path.exists(idle_path):
        return idle_path
    subprocess.run(
        [
            "ffmpeg", "-y", "-v", "error", "-i", video_path,
            "-vf", "scale=480:854,format=yuv420p",
            "-c:v", "libx264", "-preset", "veryslow", "-crf", "28",
            "-profile:v", "baseline", "-level", "3.0",
            "-an",  # 대기 루프는 항상 muted라 오디오 트랙 자체가 불필요
            "-movflags", "+faststart",  # 진행형 다운로드(Range 요청) 시작 지연 최소화
            idle_path,
        ],
        check=True,
    )
    return idle_path


@app.on_event("startup")
def _on_startup():
    load_models()
    # 알고 있는 페르소나는 서버 뜰 때 미리 준비해둬서, 첫 실제 요청부터 빠르게 만든다
    for persona in PERSONA_MAP.values():
        _prepare_persona(persona["video_path"])
        _ensure_idle_loop_video(persona["video_path"])
    print("[ML-TIMING] 서버 시작 준비 완료 - 모델/페르소나 전부 상주", flush=True)


def sh(cmd: str):
    print(">>", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash", stdin=subprocess.DEVNULL)


def get_fps(video_path: str) -> float:
    out = subprocess.check_output(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=r_frame_rate", "-of", "csv=p=0", video_path]
    ).decode().strip()
    num, den = out.split("/")
    return round(float(num) / float(den), 3)


def get_frame_size(video_path: str) -> tuple[int, int]:
    out = subprocess.check_output(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height", "-of", "csv=p=0:s=x", video_path]
    ).decode().strip()
    w, h = out.split("x")
    return int(w), int(h)


class MediaLineRequest(BaseModel):
    speaker_id: str
    speaker_name: str | None = None
    order: int | None = None
    text: str
    emotion: str | None = None


@app.get("/health")
def health():
    return {"status": "ok", "device": str(_device), "personas_ready": list(_persona_cache.keys())}


@app.post("/generate")
def generate(req: MediaLineRequest):
    persona = PERSONA_MAP.get(req.speaker_id)
    if not persona:
        raise HTTPException(400, f"unknown speaker_id: {req.speaker_id}")

    job_id = str(int(time.time() * 1000))
    audio_mp3 = f"data/audio/{job_id}.mp3"
    audio_wav = f"data/audio/{job_id}.wav"
    frame_dir = f"results/{job_id}/frames"
    os.makedirs(frame_dir, exist_ok=True)

    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice=persona["voice"],
        input=req.text,
        instructions=persona["instructions"],
    ) as response:
        response.stream_to_file(audio_mp3)
    sh(f"ffmpeg -y -v error -i {audio_mp3} {audio_wav}")

    # FastAPI가 동기 def 라우트는 자동으로 스레드풀에서 돌려주므로 별도 executor 불필요
    fps = run_inference_sync(persona["video_path"], audio_wav, frame_dir)

    temp_vid_path = f"results/{job_id}/temp.mp4"
    out_path = f"results/{job_id}/{job_id}.mp4"
    sh(f"ffmpeg -y -v warning -r {fps} -f image2 -i {frame_dir}/%08d.png "
       f"-vcodec libx264 -vf format=yuv420p -crf 18 {temp_vid_path}")
    sh(f"ffmpeg -y -v warning -i {audio_wav} -i {temp_vid_path} {out_path}")

    shutil.rmtree(frame_dir)
    os.remove(temp_vid_path)

    if not os.path.exists(out_path):
        raise HTTPException(500, "lipsync generation failed")

    return FileResponse(out_path, media_type="video/mp4", filename=f"{job_id}.mp4")


@app.websocket("/generate-stream")
async def generate_stream(ws: WebSocket):
    """모델은 서버 시작 시 이미 상주 - 요청마다 새 프로세스를 띄우지 않고, 이미 로드된
    모델로 스레드풀에서 추론 함수를 직접 호출한다.

    스트리밍 방식: 프레임을 청크 단위로 잘라서 독립된 mp4 파일을 여러 개 만드는 대신,
    프레임이 생기는 대로 순서대로 하나의 연속된 fragmented MP4(fMP4) muxing
    프로세스(ffmpeg)에 흘려보내고, 그 출력 바이트를 그대로 WebSocket으로 전달한다.
    프론트엔드는 MediaSource(SourceBuffer.appendBuffer)로 이어붙여서 재생하므로
    청크 경계에서 디코더가 재시작되지 않아 끊김이 없다 (web_test_stream_mse.html 참고).

    중요: 프레임을 muxer(ffmpeg) stdin에 흘려보내는 걸 asyncio 폴링으로 하면, GPU
    추론(별도 스레드)이 CPU 작업(PIL 블렌딩 등)으로 GIL을 오래 붙잡을 때 asyncio
    이벤트 루프가 스케줄링을 못 받아 몇 초~몇십 초씩 멈추는 문제가 있었다(실측 확인).
    그래서 muxer로의 입력은 추론과 "같은 스레드" 안에서 프레임이 생기는 즉시 동기적으로
    쓰고(frame_callback), 출력 읽기는 전용 OS 스레드가 블로킹 read로 큐에 밀어넣어
    이벤트 루프는 그 큐를 소비만 하도록 분리했다."""
    await ws.accept()
    t0 = time.time()

    def elapsed():
        return round(time.time() - t0, 2)

    def log_timing(label):
        print(f"[TIMING] {label}: {elapsed()}s", flush=True)

    mux_proc = None
    try:
        req = await ws.receive_json()
        persona = PERSONA_MAP.get(req.get("speaker_id"))
        if not persona:
            await ws.send_json({"type": "error", "message": f"unknown speaker_id: {req.get('speaker_id')}"})
            return

        job_id = str(int(time.time() * 1000))
        audio_mp3 = f"data/audio/{job_id}.mp3"
        audio_wav = f"data/audio/{job_id}.wav"
        video_path = persona["video_path"]

        FPS = get_fps(video_path)
        FRAME_W, FRAME_H = get_frame_size(video_path)
        log_timing("요청 수신 + fps/해상도 확인")

        # 1. TTS
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice=persona["voice"],
            input=req["text"],
            instructions=persona["instructions"],
        ) as response:
            response.stream_to_file(audio_mp3)
        log_timing("TTS(OpenAI) 완료")
        subprocess.run(f"ffmpeg -y -v error -i {audio_mp3} {audio_wav}", shell=True, check=True)
        log_timing("TTS mp3->wav 변환 완료")
        await ws.send_json({"type": "status", "message": "audio_ready", "elapsed": elapsed()})

        # 2. 프레임을 실시간으로 먹여서 하나의 연속 fMP4로 muxing하는 ffmpeg 프로세스.
        # image2pipe(PNG) 대신 rawvideo(압축 없는 BGR)로 입력받는다 - frame_callback이
        # 이제 PNG로 압축하지 않고 raw bytes를 바로 넘기므로, ffmpeg도 그에 맞춰
        # 압축 해제 없이 바로 읽어들이게 함 (프레임당 PNG 인코딩/디코딩 왕복 제거).
        # -g/-keyint_min/-sc_threshold로 키프레임 간격을 fps(=약 1초)로 고정해야
        # frag_keyframe이 그 주기로 fragment를 만들어 적당한 크기로 흘러나온다.
        # -profile baseline -level 3.0은 프론트엔드 MIME_CODEC(avc1.42E01E)과 반드시 일치해야 함.
        # -flush_packets 1 / -max_interleave_delta 0: mp4 muxer가 패킷을 내부에 버퍼링하지
        # 않고 즉시 파이프로 흘려보내게 강제.
        # 일반 subprocess.Popen(비-asyncio) 사용 - stdin은 추론 스레드에서, stdout은
        # 아래 전용 스레드에서 직접 블로킹 호출로 다루기 위함.
        keyint = max(1, round(FPS))
        mux_proc = subprocess.Popen(
            [
                "ffmpeg", "-y", "-v", "error",
                "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{FRAME_W}x{FRAME_H}",
                "-r", str(FPS), "-i", "pipe:0",
                "-i", audio_wav,
                # 480x854/crf28: 버퍼링 0회 확인된 안전한 값 (576x1024/crf24는 짧은 버퍼링
                # 1회 발생 - streaming_optimization.md 시도 9 참고). 화질보다 안정성 우선,
                # Cloudflare Quick Tunnel(무료, 대역폭 보장 없음)을 그대로 쓰기로 함
                "-vf", "scale=480:854,format=yuv420p",
                "-c:v", "libx264", "-preset", "ultrafast", "-tune", "zerolatency",
                "-profile:v", "baseline", "-level", "3.0",
                "-g", str(keyint), "-keyint_min", str(keyint), "-sc_threshold", "0",
                "-crf", "28",
                "-c:a", "aac", "-ar", "44100", "-ac", "2",
                "-movflags", "frag_keyframe+empty_moov+default_base_moof",
                "-max_interleave_delta", "0",
                "-flush_packets", "1",
                "-f", "mp4", "pipe:1",
            ],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        )
        log_timing("muxer(ffmpeg) 프로세스 시작")

        out_queue: "queue.Queue[bytes | None]" = queue.Queue()

        def _read_stdout():
            try:
                while True:
                    data = mux_proc.stdout.read(65536)
                    if not data:
                        break
                    out_queue.put(data)
            finally:
                out_queue.put(None)  # 종료 신호

        reader_thread = threading.Thread(target=_read_stdout, daemon=True)
        reader_thread.start()

        # 생성(GPU)은 배치 단위(16프레임)로 몰아서 끝나기 때문에 프레임이 "버스트"로
        # 나온다. 생성 속도가 실시간보다 빠르다고(1.6배속 실측) 해서 그대로 ffmpeg에
        # 밀어넣으면, 순간순간은 여전히 울퉁불퉁하게 전달돼서 클라이언트가 아주 살짝만
        # 전달이 늦어도 재생 버퍼가 바닥나 버린다. 그래서 프레임 생성(빠르게, 큐에 적재)과
        # ffmpeg로의 전달(fps에 맞춰 일정한 간격으로, 페이싱)을 분리한다 - 실시간 스트리밍
        # 서비스들이 다 쓰는 방식과 동일. 이러면 클라이언트가 필요한 버퍼 여유가 훨씬
        # 작아져서(수 초 -> 1초 미만) 재생 시작 지연도 줄어든다.
        frame_queue: "queue.Queue[bytes | None]" = queue.Queue()

        def _frame_cb(raw_bgr_bytes: bytes):
            # run_inference_sync와 같은 스레드에서 동기 호출됨 - 큐에 넣기만 하니 빠름
            frame_queue.put(raw_bgr_bytes)

        def _pace_and_feed():
            frame_interval = 1.0 / (FPS * PACE_SPEED_MULTIPLIER)
            next_t = time.time()
            while True:
                item = frame_queue.get()
                if item is None:
                    break
                wait = next_t - time.time()
                if wait > 0:
                    time.sleep(wait)
                try:
                    mux_proc.stdin.write(item)
                except (BrokenPipeError, OSError):
                    break  # muxer가 먼저 죽었으면(에러) 더 못 먹임 - stderr는 아래에서 확인
                next_t += frame_interval

        pacer_thread = threading.Thread(target=_pace_and_feed, daemon=True)
        pacer_thread.start()

        def _run_and_close_stdin():
            try:
                return run_inference_sync(video_path, audio_wav, None, _frame_cb)
            finally:
                frame_queue.put(None)  # 페이서 스레드 종료 신호
                pacer_thread.join()
                try:
                    mux_proc.stdin.close()
                except OSError:
                    pass

        # 3. 추론은 블로킹(GPU+CPU) 작업이라 스레드풀에서 실행
        loop = asyncio.get_event_loop()
        gen_future = loop.run_in_executor(EXECUTOR, _run_and_close_stdin)
        log_timing("추론 시작 (스레드풀, 모델은 이미 상주 상태)")

        # 4. 출력 큐를 소비하면서 WebSocket으로 전달 (큐 get 자체도 블로킹이라 기본
        # executor에서 실행 - 이벤트 루프는 그동안 다른 작업 처리 가능)
        # 진단용 계측: wait(ffmpeg가 다음 청크를 만들 때까지 기다린 시간 = 생성/인코딩 속도)와
        # send(ws.send_bytes 자체가 걸린 시간 = 네트워크/터널 backpressure)를 분리해서 찍는다.
        # 서버 쪽 elapsed는 짧은데 클라이언트 수신은 훨씬 오래 걸리는 현상이 있어서,
        # 그 지연이 우리 프로세스 안(생성/전송 로직)인지 그 바깥(네트워크)인지 구분하기 위함.
        first_byte_sent = False
        send_count = 0
        total_bytes = 0
        while True:
            wait_start = time.time()
            data = await loop.run_in_executor(None, out_queue.get)
            wait_dur = round(time.time() - wait_start, 3)
            if data is None:
                break
            if not first_byte_sent:
                first_byte_sent = True
                log_timing("첫 스트림 데이터 전송")
            send_start = time.time()
            await ws.send_bytes(data)
            send_dur = round(time.time() - send_start, 3)
            send_count += 1
            total_bytes += len(data)
            print(
                f"[SEND-TIMING] #{send_count} size={len(data)}B wait={wait_dur}s "
                f"send={send_dur}s total_sent={total_bytes}B elapsed={elapsed()}s",
                flush=True,
            )

        await gen_future  # 추론 스레드 안에서 예외가 났으면 여기서 다시 raise됨
        await loop.run_in_executor(None, mux_proc.wait)

        if mux_proc.returncode != 0:
            stderr = mux_proc.stderr.read().decode(errors="ignore")
            raise RuntimeError(f"muxer ffmpeg 실패(code={mux_proc.returncode}): {stderr[-1000:]}")

        log_timing("전체 완료")
        await ws.send_json({"type": "done", "elapsed": elapsed()})
    except Exception as e:
        await ws.send_json({"type": "error", "message": str(e)})
    finally:
        if mux_proc and mux_proc.poll() is None:
            mux_proc.kill()
        await ws.close()


## 12. FastAPI 서버 백그라운드 실행

Colab 셀은 명령이 끝나야 다음 셀로 넘어가므로, 서버는 `nohup ... &`로 백그라운드 실행합니다. 로그는 `uvicorn.log`에 쌓입니다.

In [ ]:
import os
# app.py가 OpenAI 클라이언트를 초기화하려면 OPENAI_API_KEY가 필요 (7번 단계에서 이미 os.environ에 설정됨)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다 - 7번 단계를 먼저 실행하세요"

get_ipython().system_raw(
    f'export OPENAI_API_KEY="{os.environ["OPENAI_API_KEY"]}" && '
    # MPLBACKEND=Agg: 마스크/랜드마크 계산(mmpose -> xtcocotools -> matplotlib)이 이제
    # 서버 프로세스 안에서 직접 실행되므로, uvicorn 프로세스 자체에 이 값이 있어야
    # Colab 커널의 인라인 백엔드를 상속받아 죽는 문제가 안 생김
    'export MPLBACKEND=Agg && '
    "source /usr/local/miniconda/etc/profile.d/conda.sh && conda activate musetalk && "
    "cd /content/MuseTalk && "
    "nohup uvicorn app:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &"
)
print("서버 백그라운드로 시작함, 몇 초 후 아래 셀로 확인")

In [ ]:
import time, subprocess
# 모델 상주 구조로 바뀌면서 서버 시작 시 모델 로드+페르소나 워밍업이 끝나야 응답하므로
# (최초 캐시가 없으면 최대 1분 가까이 걸릴 수 있음) 대기 시간을 넉넉하게 늘림
for _ in range(90):
    r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://localhost:8000/health"],
                        capture_output=True, text=True)
    if r.stdout.strip() == "200":
        print("서버 정상 기동 확인")
        break
    time.sleep(1)
else:
    print("서버가 아직 안 떴어요, uvicorn.log 확인 필요:")
    print(open("uvicorn.log").read()[-2000:])

## 13. cloudflared로 외부 노출

Colab 내부(localhost)에서만 접근되던 API를 외부(웹 프론트, curl 등)에서 접근 가능하게 임시 터널을 엽니다. 계정 가입 없이 바로 되는 "Quick Tunnel"이라 URL이 재실행마다 바뀝니다 — 세션이 끊기면 다시 이 단계부터 실행해서 새 URL을 받아야 합니다.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

get_ipython().system_raw("./cloudflared tunnel --url http://localhost:8000 > cloudflared.log 2>&1 &")
print("cloudflared 시작함, 몇 초 후 아래 셀로 URL 확인")

In [ ]:
import time, re

public_url = None
for _ in range(20):
    log = open("cloudflared.log").read()
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", log)
    if m:
        public_url = m.group(0)
        break
    time.sleep(1)

if public_url:
    print("외부 접근 URL:", public_url)
    print("health check:", public_url + "/health")
else:
    print("URL을 못 찾았어요, 로그 확인:")
    print(open("cloudflared.log").read())

## 14. 외부 URL로 실제 테스트

`public_url`(13번에서 받은 주소)로 `/generate`를 호출해 실제로 립싱크 영상이 응답으로 오는지 확인합니다. 이게 되면 프론트엔드(가은님 쪽)에서도 이 URL로 똑같이 호출해서 영상을 받아볼 수 있다는 뜻입니다.

In [ ]:
import requests, time

payload = {
    "speaker_id": "business_strategy",
    "speaker_name": "사업전략 전문가",
    "order": 1,
    "text": "안녕하세요. 사업 전략 관점에서 이번 계획서를 검토했습니다.",
    "emotion": "serious",
}

t0 = time.time()
resp = requests.post(public_url + "/generate", json=payload, timeout=300)
print("status:", resp.status_code, f"({time.time() - t0:.1f}초 소요)")

if resp.status_code == 200:
    with open("api_test_result.mp4", "wb") as f:
        f.write(resp.content)
    print("영상 저장됨: api_test_result.mp4 (", len(resp.content), "bytes )")
else:
    print(resp.text)